# 🛠️ Chapter 03-02b: Hands-On — LCEL Multi-Step Analysis Pipeline

---

## Time to Compose Like a Pro! 🎵

In this notebook, you'll master every Runnable primitive and build a real multi-step analysis pipeline.

By the end:
- ✅ Used RunnablePassthrough to forward data
- ✅ Transformed data with RunnableLambda
- ✅ Run parallel chains with RunnableParallel
- ✅ Streamed responses in real-time
- ✅ Batch-processed multiple inputs
- ✅ Built a multi-step article analysis pipeline

Let's compose! 🚀

---

## 📦 Setup

In [ ]:
# !pip install langchain-core langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("✅ Ready!" if os.getenv("OPENAI_API_KEY") else "❌ Set OPENAI_API_KEY in .env")

---

## 🧪 Exercise 1: RunnablePassthrough

In [ ]:
# RunnablePassthrough passes input through unchanged

passthrough = RunnablePassthrough()
result = passthrough.invoke("Hello, I pass through!")
print(f"Passthrough: '{result}'")

# With dict input
result = passthrough.invoke({"name": "Sarthak", "topic": "AI"})
print(f"Dict passthrough: {result}")

In [ ]:
# .assign() — keep original data AND add computed fields

def get_length(data):
    return len(data["text"].split())

chain = RunnablePassthrough.assign(
    word_count=RunnableLambda(get_length)
)

result = chain.invoke({"text": "LangChain is a powerful framework for building AI apps"})
print(f"Result: {result}")
print(f"💡 Original 'text' key preserved + new 'word_count' added!")

---

## 🧪 Exercise 2: RunnableLambda

In [ ]:
# Wrap any Python function as a Runnable

def clean_text(text: str) -> str:
    """Custom preprocessing step."""
    return text.strip().lower().replace("\n", " ")

def add_metadata(text: str) -> dict:
    """Add metadata to processed text."""
    return {
        "text": text,
        "char_count": len(text),
        "word_count": len(text.split())
    }

# Chain custom functions together
preprocess_chain = RunnableLambda(clean_text) | RunnableLambda(add_metadata)

result = preprocess_chain.invoke("  Hello WORLD!\n  This is a TEST.  ")
print(f"Preprocessed: {result}")

In [ ]:
# RunnableLambda in a full chain with LLM

prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the text in exactly one sentence."),
    ("human", "{text}")
])

def format_output(text: str) -> str:
    return f"📝 Summary: {text}"

chain = (
    {"text": RunnableLambda(clean_text)}  # Preprocess
    | prompt                               # Format
    | llm                                  # Generate
    | StrOutputParser()                    # Extract text
    | RunnableLambda(format_output)        # Post-process
)

result = chain.invoke("  Machine learning is a SUBSET of artificial intelligence.\n  It enables computers to LEARN from data.  ")
print(result)

---

## 🧪 Exercise 3: RunnableParallel

In [ ]:
# Run multiple chains in PARALLEL on the same input

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize this text in 1 sentence."),
    ("human", "{text}")
])

keywords_prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract 5 keywords from this text. Return only comma-separated keywords."),
    ("human", "{text}")
])

sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the sentiment of this text as: positive, negative, or neutral. Return only the word."),
    ("human", "{text}")
])

# All three chains run at the SAME TIME
parallel_chain = RunnableParallel(
    summary=summary_prompt | llm | StrOutputParser(),
    keywords=keywords_prompt | llm | StrOutputParser(),
    sentiment=sentiment_prompt | llm | StrOutputParser(),
)

text = """OpenAI released GPT-4o, a multimodal model that can process text, images, and audio. 
It's faster and cheaper than GPT-4 Turbo, making advanced AI more accessible to developers. 
The model shows impressive performance on coding, math, and creative tasks."""

import time
start = time.time()
result = parallel_chain.invoke({"text": text})
elapsed = time.time() - start

print("⚡ PARALLEL ANALYSIS (all 3 chains ran simultaneously):")
print(f"   Summary:   {result['summary']}")
print(f"   Keywords:  {result['keywords']}")
print(f"   Sentiment: {result['sentiment']}")
print(f"\n   ⏱️ Time: {elapsed:.2f}s (would be ~3× slower sequentially!)")

---

## 🧪 Exercise 4: Streaming Responses

In [ ]:
# Streaming: see tokens appear one by one

story_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a storyteller. Tell a very short story (3-4 sentences)."),
    ("human", "Tell a story about {topic}")
])

story_chain = story_prompt | llm | StrOutputParser()

print("📖 Streaming story about AI:\n")
for chunk in story_chain.stream({"topic": "a robot learning to paint"}):
    print(chunk, end="", flush=True)

print("\n\n💡 Each chunk is a few characters/tokens. Feels instant to users!")

---

## 🧪 Exercise 5: Batch Processing

In [ ]:
# Batch: process multiple inputs in parallel

explain_prompt = ChatPromptTemplate.from_messages([
    ("system", "Explain the concept in exactly one sentence for a beginner."),
    ("human", "Explain: {concept}")
])

explain_chain = explain_prompt | llm | StrOutputParser()

# Process 5 concepts at once
concepts = [
    {"concept": "Machine Learning"},
    {"concept": "Neural Networks"},
    {"concept": "Transformers"},
    {"concept": "Embeddings"},
    {"concept": "RAG"},
]

start = time.time()
results = explain_chain.batch(concepts)
elapsed = time.time() - start

print("📦 BATCH RESULTS:")
for concept, result in zip(concepts, results):
    print(f"   {concept['concept']}: {result}")

print(f"\n   ⏱️ Total time for 5 calls: {elapsed:.2f}s")
print(f"   (Sequential would be ~{elapsed * 3:.1f}s+)")

---

## 🚀 Build: Multi-Step Article Analysis Pipeline

Let's combine everything into a real analysis system!

In [ ]:
# Multi-step pipeline:
# Input → Preprocess → Parallel(Summarize, Extract Keywords, Classify) → Combine → Final Report

# Step 1: Preprocessing
def preprocess(data: dict) -> dict:
    text = data["text"].strip()
    return {"text": text, "word_count": len(text.split())}

# Step 2: Analysis chains
summary_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize in 2 sentences."),
        ("human", "{text}")
    ]) | llm | StrOutputParser()
)

keywords_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Return exactly 5 keywords, comma-separated."),
        ("human", "{text}")
    ]) | llm | StrOutputParser()
)

category_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Classify into: Technology, Science, Business, Health, or Other. Return only the category."),
        ("human", "{text}")
    ]) | llm | StrOutputParser()
)

# Step 3: Parallel analysis
analysis = RunnableParallel(
    summary=summary_chain,
    keywords=keywords_chain,
    category=category_chain,
    text=RunnablePassthrough() | RunnableLambda(lambda x: x["text"]),
)

# Step 4: Format report
def format_report(data: dict) -> str:
    return f"""📄 ARTICLE ANALYSIS REPORT
{'=' * 50}
📂 Category:  {data['category']}
🏷️ Keywords:  {data['keywords']}
📝 Summary:   {data['summary']}
{'=' * 50}"""

# Full pipeline
pipeline = (
    RunnableLambda(preprocess)
    | analysis
    | RunnableLambda(format_report)
)

# Test it!
article = """Anthropic has released Claude 3.5 Sonnet, which outperforms GPT-4o on several 
benchmarks while being significantly cheaper. The model excels at coding tasks and can 
handle complex multi-step reasoning. This marks a significant shift in the competitive 
landscape of large language models, with Anthropic establishing itself as a serious 
contender against OpenAI and Google."""

result = pipeline.invoke({"text": article})
print(result)

---

## ✍️ Practice: Build a Multi-Language Summary System

**Challenge:** Take an article, summarize it, then translate the summary into 3 languages in parallel.

In [ ]:
# YOUR CODE HERE

# TODO: Create a summary chain
# TODO: Create 3 translation chains (Hindi, Spanish, French)
# TODO: Use RunnableParallel to translate simultaneously
# TODO: Chain: article → summary → parallel translations

# Test with any article text

<details>
<summary>💡 Click to see solution</summary>

```python
summary_step = ChatPromptTemplate.from_messages([
    ("system", "Summarize in 1 sentence."),
    ("human", "{text}")
]) | llm | StrOutputParser()

def make_translate_chain(lang):
    return ChatPromptTemplate.from_messages([
        ("system", f"Translate to {lang}. Return only the translation."),
        ("human", "{text}")
    ]) | llm | StrOutputParser()

multi_translate = RunnableParallel(
    hindi=make_translate_chain("Hindi"),
    spanish=make_translate_chain("Spanish"),
    french=make_translate_chain("French"),
)

full_chain = (
    {"text": RunnablePassthrough()}
    | RunnableLambda(lambda x: {"text": summary_step.invoke(x)})
    | multi_translate
)

result = full_chain.invoke(article)
for lang, text in result.items():
    print(f"{lang}: {text}")
```

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 RunnablePassthrough: forward data, .assign() adds keys.       │
├─────────────────────────────────────────────────────────────────────┤
│  📌 RunnableLambda: wrap Python functions for LCEL chains.        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 RunnableParallel: concurrent execution = speed boost.         │
├─────────────────────────────────────────────────────────────────────┤
│  📌 .stream(): real-time token output. .batch(): parallel inputs.  │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Complex pipelines = combine simple steps with |               │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Pre-process → Parallel Analysis → Post-process = real pattern  │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Tools & Output Parsers!

Output parsers for structured data, tool definitions, and building an LLM that can search the web!

---

### 🎉 You're Composing Like a Pro!
Passthrough, Lambda, Parallel, Streaming, Batching — you own the LCEL toolkit! 🚀